# 10 (PH) - The model on AWS: the same script as SageMaker training jobs, the search as tuning jobs

**Who worked on this notebook**

| Who | What |
| --- | --- |
| Peter Henskens | Decided after 06 that the model is finally built and trained on AWS; holds the AWS Academy Learner Lab account and its budget; confirmed the widened spaces of 08 §4, the instance type and the cost before the run; ran the notebook on 10 October (five smoke jobs, two tuning jobs, the refit), pasted the lab's credentials each session, reviewed each job and confirmed the tie-break of part 6 |
| Claude Code (AI assistant) | Drafted `citibike/train/jobs.py` (the requests, the waits, the artefacts) and its tests, the two changes to `train.py` for the container, the notebook, the checks and the dry run of 8 October (appendix). Logged in `genai-log.md` |

**Goal.** 07 wrote the training as a script so that it could leave the laptop. This notebook sends that script to
SageMaker **unchanged**: first one smoke job on a station sample that proves the container, the packages, the
channels and the metric line; then the search of 08 as two **hyperparameter tuning jobs** on all rows, in the
spaces 08 §4 widened where 08's winners sat on the edge; then the best configuration per horizon as a training job
whose artefacts come back to `models/citibike/10_<horizon>_<family>/`, where 09 scores them next to 05, 07 and 08 on
the test period, once, with the rule it fixed. The AWS account is the AWS Academy Learner Lab: a real account with a
budget, two regions and sessions of four hours.

**Products**

| Product | What it is |
| --- | --- |
| `s3://<bucket>/citibike/model_inputs/<horizon>/{train,valid}/` | the model inputs of 07 §2 as the jobs' channels (the test file stays home) |
| `s3://<bucket>/citibike/source/sourcedir.tar.gz` | `train.py` and its `requirements.txt`: what the container runs |
| the jobs in SageMaker | one smoke job, two tuning jobs (12 + 6 training jobs), one refit; names `citibike-<tag>-...`, their artefacts under `s3://<bucket>/citibike/jobs/` |
| `models/citibike/10_search/<horizon>_trials.csv` | every trial with its hyperparameters, validation MAE and time |
| `models/citibike/10_<horizon>_<family>/` | `model.joblib`, `metrics.json`, `predictions_valid.parquet` and `job.json` of the winner per horizon, for 09 |

**How it is built**
0. Setup: the lab, the credentials, the role.
1. The design: script mode through the plain API, the container, what the lab allows, the search, the cost.
2. To S3: the bucket, the inputs, the source bundle.
3. The smoke job.
4. Horizon day: the tuning job and its trials.
5. Horizon rest of month: the grid as a tuning job, the refit.
6. The winners back on the laptop, checked.
7. Reading and what carries forward.
Appendix: the dry run of 8 October, the expected runtime and cost, the failures to expect.

**Status: run against the AWS Academy Learner Lab on 10 October 2026 (team lead).** The requests were tested against
the AWS API model (`citibike/tests/test_jobs.py`) and the dry run of the appendix ran every cell against a fake AWS
first; four smoke jobs failed on the container's packages before the fifth passed (appendix). The reading in part 7
is written from the outputs of the run.

## 0. Setup: the lab, the credentials, the role

Kernel "DataSpell", as 07 - 09; `boto3` is new in `requirements.txt`. **The credentials** change with every lab
session and live in the git-ignored `.env` in the repo root, next to the database login (`.env.example` lists the
keys): on the lab page, *AWS Details → AWS CLI → Show* gives `aws_access_key_id`, `aws_secret_access_key` and
`aws_session_token`, which go in as `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` and `AWS_SESSION_TOKEN`. Nothing of
that goes into this notebook or a chat; the setup cell loads the `.env` (overriding what an earlier session left in
the kernel), checks that boto3 found credentials there (or in `~/.aws/credentials`, which boto3 also reads) and
prints the role it runs as and the last four digits of the account, never a key. **The proxy**: the managed laptop
re-signs HTTPS traffic to S3 with its own certificate, which Python's own CA bundle does not know; the first two lines
of the cell make Python trust the operating system's certificate store instead, as `citibike_data.py` does for the
downloads. They must run before `boto3` is imported, so after adding them to a running kernel, restart it.
The region is us-east-1 (the lab allows us-east-1 and us-west-2). **The LabRole** is the execution role of every
job: the lab created it with the rights SageMaker and S3 need; IAM is locked down, so it is not looked up but named
from the account id. The bucket is the name the SageMaker SDK would use, created in part 2 when missing.

In [1]:
import truststore                 # first of all, before boto3: TLS through the operating system's certificate store, because the
truststore.inject_into_ssl()      # laptop's proxy re-signs S3 traffic (citibike_data.py does the same); harmless on any other machine

import json
import os
import sys
import time
from contextlib import redirect_stdout
from pathlib import Path

import boto3
import pandas as pd
from dotenv import find_dotenv, load_dotenv

sys.path.insert(0, "..")         # citibike/ -> connection.py and modelling.py
sys.path.insert(0, "../train")   # citibike/train/ -> train.py (the script) and jobs.py (the requests)
import jobs
import train
from connection import fmt, notebook_style
from modelling import load_features, scorable, run_predictions, model_predictions, MODEL_INPUTS_DIR, MODELS_DIR

notebook_style()
print(f"python {sys.version.split()[0]}, boto3 {boto3.__version__}, pandas {pd.__version__}")

REGION = "us-east-1"
RUN_TAG = "r5"                   # part of every job name; names are unique per account, so a fresh set of jobs needs a new tag (r1 - r4: the package failures of 10 October)
HORIZONS = ["day", "rest_of_month"]
FAMILY = {"day": "lightgbm", "rest_of_month": "extra_trees"}         # 06's choice, as 07 and 08
INSTANCE = "ml.m5.xlarge"        # 4 vCPU, 16 GB: the largest the lab allows for SageMaker (medium, large, xlarge)
TRAIN_DIR = Path("../train")
SEARCH_DIR = MODELS_DIR / "10_search"
RUNS_10 = {horizon: MODELS_DIR / f"10_{horizon}_{family}" for horizon, family in FAMILY.items()}

load_dotenv(find_dotenv(usecwd=True), override=True)   # the .env in the repo root; override: a new session's values replace an old session's
session = boto3.Session(region_name=REGION)
credentials = session.get_credentials()
assert credentials is not None, ("no AWS credentials found: start the lab, AWS Details -> AWS CLI -> Show, and put the three values in the "
                                 "repo's .env as AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY and AWS_SESSION_TOKEN")
assert credentials.method in ("env", "shared-credentials-file"), f"the credentials come from '{credentials.method}'; this notebook takes them from .env (or ~/.aws/credentials) only"
who = jobs.identity(session)
sm, s3 = session.client("sagemaker"), session.client("s3")
BUCKET = f"sagemaker-{REGION}-{who['account']}"
ROLE_ARN = os.environ.get("SAGEMAKER_ROLE_ARN") or who["role_arn"]     # .env may name the role this lab lets you pass (part 1)
print(f"credentials from {'.env' if credentials.method == 'env' else '~/.aws/credentials'}; account {jobs.masked(who['account'])}, "
      f"running as {who['caller']}, region {REGION}, execution role {ROLE_ARN.split('/')[-1]}"
      f"{' (from .env)' if 'SAGEMAKER_ROLE_ARN' in os.environ else ' (the lab default)'}, bucket {BUCKET}")
in_use = jobs.roles_in_use(sm)
print("roles this account already hands to SageMaker: " + (", ".join(r.split("/")[-1] for r in in_use) if in_use else "none visible"))
for horizon in HORIZONS:
    assert (MODEL_INPUTS_DIR / horizon / "valid.parquet").exists(), f"run 07 §2 first: no model inputs for {horizon}"

python 3.13.7, boto3 1.43.109, pandas 2.3.3
credentials from .env; account ********6158, running as voclabs, region us-east-1, execution role LabRole (the lab default), bucket sagemaker-us-east-1-387312886158
roles this account already hands to SageMaker: none visible


## 1. The design

**Decision: script mode through the plain AWS API (boto3), not the SageMaker Python SDK.** Three roads were open:
(A) the SDK's `SKLearn` estimator and `HyperparameterTuner`, the usual road, but its current major version (3.x)
would install torch and mlflow next to this kernel, and its calls hide the request; (B) **boto3 with the requests
written out** in `citibike/train/jobs.py`: every field of a training job and a tuning job is on the page, nothing is
installed but boto3, and the requests are tested against the AWS API model without credentials; (C) SageMaker's
built-in algorithms: another family than 06 chose (07 §1). B. What the container does with a request, and what the
script sees:

| In the request | In the container |
| --- | --- |
| `TrainingImage`: the scikit-learn framework image, `1.2-1-cpu-py3` | Python 3.9 with numpy 1.24.1, scipy 1.8.0, pandas 1.1.3, pyarrow 14.0.1 and scikit-learn 1.2.1 installed by conda; it runs `pip install -r requirements.txt` from the bundle first, which must leave those compiled packages alone and only add scikit-learn 1.3 (the `TargetEncoder`) and lightgbm |
| the hyperparameters `sagemaker_program` and `sagemaker_submit_directory` | which script to run, and the S3 bundle to fetch it from |
| every other hyperparameter, as a JSON string | one `--name value` argument of `train.py`, decoded (`"0.05"` is a number again) |
| the channels `train` and `valid`, each an S3 prefix with one Parquet file | the directories `SM_CHANNEL_TRAIN` and `SM_CHANNEL_VALID`, which `read_inputs()` reads |
| `OutputDataConfig` | `SM_MODEL_DIR` packed as `model.tar.gz`, `SM_OUTPUT_DATA_DIR` (metrics.json, predictions) as `output.tar.gz` |
| `MetricDefinitions`: a regex per metric | the `validation:MAE=...` line the script prints becomes the job's metric, and the tuner's objective |

**Two changes to the script for the container, nothing else.** Python 3.9 does not evaluate `list[str] | None` in a
signature, so `train.py` got `from __future__ import annotations`; and the script's `requirements.txt` pins numpy,
scipy, pandas and pyarrow to exactly what the container has (1.24.1, 1.8.0, 1.1.3, 14.0.1) and adds only scikit-learn
1.3.2 and lightgbm 4.5. Four smoke jobs on 10 October taught why: the container's packages were installed with conda,
and when pip replaces one of them (scipy in the third job, pyarrow in the fourth) its compiled files stay behind and
the new version loads them (`numpy.core.multiarray failed to import`, a TypeError inside scipy, `_multiarray_umath
failed to import`). The set was installed and run on a clean Python 3.9 on the laptop first. 07 and 08 run the same
file as before (the dry run of the appendix).

**What the lab allows** (the Learner Lab README of June 2025): us-east-1 and us-west-2; SageMaker "can assume the
LabRole IAM role"; SageMaker instance types `ml.t3.medium/large/xlarge`, `ml.m5.large/xlarge`, `ml.c5.large/xlarge`
only, so `ml.m5.xlarge` (4 vCPU, 16 GB) is the machine of every job; at most 2 notebook instances and 2 Studio apps
(this notebook creates neither); the environment is long-lived (the bucket, the jobs and their artefacts stay
between sessions) and **exceeding the lab budget disables the account and deletes everything in it**, which is why
every cell prints the billed seconds and the jobs are few and small. The LabRole as the execution role, **or the
role the lab names**: a guided lab (the challenge lab of the Machine Learning Foundations course) may
refuse `iam:PassRole` on LabRole and allow only the role its own SageMaker notebook instance runs with; the setup
cell prints the roles the account already hands to SageMaker, and `SAGEMAKER_ROLE_ARN` in `.env` makes the notebook
use one of them; a session of four hours, after which the credentials expire while the jobs keep running: every cell that waits
finds its job by name and never submits it twice, so it is run again after a new session; a budget, shown on the lab
page, that the jobs below do not come near.

**The search, and what differs from 08.** *Day*: a **Bayesian** search of 12 jobs, 2 in parallel, over 08's space
widened where 08's winner sat on the edge: `num-leaves` 63 - 1.023 (08 stopped at 255), `min-child-samples`
100 - 5.000 (08 at 1.000), 1.000 trees (08's 500, with the curve still falling), the objective as a categorical axis
(08 found the three within 0,1 of each other at 255 leaves), learning rate, row and column shares as in 08. The
tuner minimises the validation MAE the script prints; the curve every 100 trees stays in each job's metrics.json.
Bayesian rather than random because each job costs minutes and the search runs two at a time anyway: the tuner
proposes the next points from the finished ones. *Rest of month*: 08's grid shifted down and across,
`min-samples-leaf` 25 / 50 / 100 and `max-features` 0,7 / 1,0 at 100 trees, as a tuning job with the **Grid**
strategy (every combination, 6 jobs); the winner refitted with 200 trees as one more training job, as 08 did on the
laptop. The benchmark is 08 on the same rows and metric: 9,634 (500 trees) and 108,47.

**Cost, an estimate.** ml.m5.xlarge is about 0,23 $ per training hour (list price, us-east-1). The smoke job takes
minutes; a day job 5 - 10 minutes (the package install first, then 1.000 trees on 4 vCPU and the curve); an extra
trees job 10 - 25 minutes, the leaf of 25 the longest; so about 5 instance-hours, **a few dollars**, over two to
three hours of wall time with two jobs in parallel. The cells print the billable seconds of every job.

In [2]:
SMOKE = {"n-estimators": 100, "curve-every": 0, "sample-stations": 0.05}       # 5 % of the stations: the first job
STATIC_DAY = {"n-estimators": 1000, "curve-every": 100}
SPACE_DAY = dict(
    continuous_ranges=[jobs.continuous("learning-rate", 0.02, 0.2, "Logarithmic"),
                       jobs.continuous("subsample", 0.5, 1.0), jobs.continuous("colsample-bytree", 0.5, 1.0)],
    integer_ranges=[jobs.integer("num-leaves", 63, 1023, "Logarithmic"), jobs.integer("min-child-samples", 100, 5000, "Logarithmic")],
    categorical_ranges=[jobs.categorical("objective", ["regression", "regression_l1", "poisson"])])
MAX_JOBS_DAY, MAX_PARALLEL = 12, 2
STATIC_REST = {"n-estimators": 100}
SPACE_REST = dict(categorical_ranges=[jobs.categorical("min-samples-leaf", [25, 50, 100]), jobs.categorical("max-features", [0.7, 1.0])])
TREES_FINAL_REST = 200
BENCHMARK_08 = {"day": 9.634, "rest_of_month": 108.47}                           # 08 §4, the same validation rows
JOB = {"smoke": f"citibike-{RUN_TAG}-smoke-day", "day": f"citibike-{RUN_TAG}-day-tune",
       "rest_of_month": f"citibike-{RUN_TAG}-rest-tune", "refit": f"citibike-{RUN_TAG}-rest-refit"}
print("\n".join(f"{kind}: {name}" for kind, name in JOB.items()))

smoke: citibike-r5-smoke-day
day: citibike-r5-day-tune
rest_of_month: citibike-r5-rest-tune
refit: citibike-r5-rest-refit


## 2. To S3: the bucket, the inputs, the source bundle

The bucket `sagemaker-<region>-<account>` (the name the SDK uses; created when it is not there), the four input
files to their channel prefixes, one file per prefix because a channel is a directory, skipped when an object of the
same size is already there (230 MB over the line once), and the bundle with the script and its requirements, always
(it is small and the script may have changed). The table says what went where.

In [3]:
created = jobs.ensure_bucket(s3, BUCKET, REGION)
print(f"bucket {BUCKET}: {'created' if created else 'exists'}")
t0 = time.time()
uploads = jobs.upload_inputs(s3, BUCKET, MODEL_INPUTS_DIR, HORIZONS)
SOURCE_URI = jobs.upload_source(s3, BUCKET, TRAIN_DIR)
print(f"{uploads.MB.sum():.0f} MB of inputs checked or uploaded in {time.time() - t0:.0f} s; the bundle at {SOURCE_URI}")
uploads

bucket sagemaker-us-east-1-387312886158: exists
228 MB of inputs checked or uploaded in 1 s; the bundle at s3://sagemaker-us-east-1-387312886158/citibike/source/sourcedir.tar.gz


,horizon,split,MB,s3,status
0,day,train,"75,8",s3://sagemaker-us-east-1-387312886158/citibike/model_inputs/day/train/train.parquet,already there
1,day,valid,"22,7",s3://sagemaker-us-east-1-387312886158/citibike/model_inputs/day/valid/valid.parquet,already there
2,rest_of_month,train,"97,9",s3://sagemaker-us-east-1-387312886158/citibike/model_inputs/rest_of_month/train/train.parquet,already there
3,rest_of_month,valid,"31,9",s3://sagemaker-us-east-1-387312886158/citibike/model_inputs/rest_of_month/valid/valid.parquet,already there


## 3. The smoke job

One training job: the day horizon on 5 % of the stations with 100 trees, on the lab's instance. It proves, in a few
minutes and cents, that the container starts, installs the requirements, finds the channels, runs `train.py` as the
laptop does, prints the metric line the tuner will read and packs the artefacts. The wait prints each change of
status; the artefacts come back to `10_search/smoke/`, and its metrics.json tells which Python and packages the
container ran. The last lines of the job's log show the install and the metric line (or the traceback, if any).

In [4]:
request = jobs.training_job_request(JOB["smoke"], ROLE_ARN, REGION, BUCKET, "day", FAMILY["day"], SMOKE, SOURCE_URI,
                                    instance_type=INSTANCE, max_runtime_s=1800)
jobs.submit(sm, request)
smoke = jobs.wait(sm, JOB["smoke"])
failed = smoke["TrainingJobStatus"] != "Completed"
print("\n".join("  | " + line for line in jobs.log_tail(session, JOB["smoke"], 80 if failed else 12)))   # the install and the traceback when it failed
assert not failed, smoke.get("FailureReason", "")[:300]
record = jobs.fetch_artefacts(s3, smoke, SEARCH_DIR / "smoke")
metrics = json.loads((SEARCH_DIR / "smoke" / "metrics.json").read_text(encoding="utf-8"))
print(f"container: python {metrics['versions']['python']}, scikit-learn {metrics['versions']['scikit-learn']}, "
      f"lightgbm {metrics['versions'].get('lightgbm')}, pandas {metrics['versions']['pandas']}; "
      f"{fmt(metrics['rows']['train'])} training rows, fit {metrics['fit_seconds']} s, job {record['training_seconds']:.0f} s, "
      f"billed {record['billable_seconds']} s; validation MAE {metrics['validation']['MAE']:.3f}, "
      f"as the metric line read it: {record['metrics'].get('validation:MAE')}")

submitted training job citibike-r5-smoke-day
08:51:15 citibike-r5-smoke-day: InProgress Starting
08:52:16 citibike-r5-smoke-day: InProgress Downloading
08:53:17 citibike-r5-smoke-day: InProgress Training
08:54:17 citibike-r5-smoke-day: Completed Completed
  | SM_HP_SAMPLE-STATIONS=0.05
  | PYTHONPATH=/opt/ml/code:/miniconda3/bin:/miniconda3/lib/python39.zip:/miniconda3/lib/python3.9:/miniconda3/lib/python3.9/lib-dynload:/miniconda3/lib/python3.9/site-packages:/miniconda3/lib/python3.9/site-packages/setuptools/_vendor
  | Invoking script with the following command:
  | /miniconda3/bin/python train.py --curve-every 0 --family lightgbm --horizon day --n-estimators 100 --sample-stations 0.05
  | 2026-10-10 06:53:13,059 sagemaker-training-toolkit INFO     Exceptions not imported for SageMaker Debugger as it is not installed.
  | 2026-10-10 06:53:13,059 sagemaker-training-toolkit INFO     Exceptions not imported for SageMaker TF as Tensorflow is not installed.
  | lightgbm day: 124,085 train

## 4. Horizon day: the tuning job and its trials

The request: the smoke request on all rows, the static hyperparameters (1.000 trees, the curve), the ranges of part
1, Bayesian, 12 jobs, 2 in parallel, no early stopping (the script reports one number, at the end). Submitted once;
the wait reads the tuning job's counters. About an hour; the cell can be interrupted and run again, it finds the job
by name. Then every trial as a row: the tuned values, the validation MAE, the training time, sorted on the MAE, saved
to `10_search/day_trials.csv`, with 08's best on the same rows for scale. What to read: whether the tuner went
beyond 08's edges (leaves above 255, a leaf floor above 1.000) and whether that paid.

In [5]:
base_day = jobs.training_job_request(JOB["day"], ROLE_ARN, REGION, BUCKET, "day", FAMILY["day"], STATIC_DAY, SOURCE_URI,
                                     instance_type=INSTANCE, max_runtime_s=7200)
request_day = jobs.tuning_job_request(JOB["day"], base_day, **SPACE_DAY, strategy="Bayesian", max_jobs=MAX_JOBS_DAY, max_parallel=MAX_PARALLEL)
jobs.submit(sm, request_day)
tuning_day = jobs.wait(sm, JOB["day"], tuning=True)
trials_day = jobs.trials(sm, JOB["day"])
SEARCH_DIR.mkdir(parents=True, exist_ok=True)
trials_day.to_csv(SEARCH_DIR / "day_trials.csv")
done_day = trials_day[trials_day.status == "Completed"]
assert len(done_day), "no day trial completed: read the failure column and the log of one job (jobs.log_tail)"
print(f"{len(trials_day)} jobs, {len(done_day)} completed, {fmt(trials_day['training (s)'].sum() / 3600, '.1f')} instance-hours; "
      f"best {done_day.MAE.iloc[0]:.3f} at 1.000 trees against 08's {BENCHMARK_08['day']} at 500 on the same rows")
trials_day.round(3)

submitted tuning job citibike-r5-day-tune
08:57:55 citibike-r5-day-tune: InProgress
08:58:56 citibike-r5-day-tune: InProgress inprogress 2
09:14:10 citibike-r5-day-tune: InProgress completed 1, inprogress 2
09:16:12 citibike-r5-day-tune: InProgress completed 2, inprogress 2
09:19:15 citibike-r5-day-tune: InProgress completed 3, inprogress 1
09:20:16 citibike-r5-day-tune: InProgress completed 3, inprogress 2
09:30:25 citibike-r5-day-tune: InProgress completed 4, inprogress 2
09:33:28 citibike-r5-day-tune: InProgress completed 5, inprogress 1
09:34:29 citibike-r5-day-tune: InProgress completed 5, inprogress 2
09:40:34 citibike-r5-day-tune: InProgress completed 6, inprogress 2
09:53:47 citibike-r5-day-tune: InProgress completed 7, inprogress 2
09:57:51 citibike-r5-day-tune: InProgress completed 8, inprogress 2
10:05:59 citibike-r5-day-tune: InProgress completed 9, inprogress 2
10:10:02 citibike-r5-day-tune: InProgress completed 10, inprogress 2
10:16:08 citibike-r5-day-tune: InProgress co

,status,colsample-bytree,learning-rate,min-child-samples,num-leaves,objective,subsample,MAE,training (s),failure
job,,,,,,,,,,
citibike-r5-day-tune-002-afd67bf2,Completed,"0,784","0,023",142,1.017,regression_l1,"0,860","9,534",1.032,
citibike-r5-day-tune-012-a929d019,Completed,"0,842","0,022",100,501,regression_l1,"0,636","9,534",868,
citibike-r5-day-tune-007-ca9660d1,Completed,"0,784","0,022",792,1.017,regression_l1,"0,654","9,566",1.387,
citibike-r5-day-tune-008-75ebf5df,Completed,"0,755","0,042",1.979,149,regression_l1,"0,820","9,609",661,
citibike-r5-day-tune-011-a51be8b4,Completed,"0,945","0,075",100,349,regression_l1,"0,593","9,679",586,
citibike-r5-day-tune-004-46395b42,Completed,"0,967","0,049",336,575,regression,"0,780","9,696",1.023,
citibike-r5-day-tune-006-05186bee,Completed,"0,596","0,023",143,81,regression_l1,"0,864","9,720",501,
citibike-r5-day-tune-009-fc13f502,Completed,"1,000","0,115",100,1.023,poisson,"0,926","9,732",963,
citibike-r5-day-tune-001-dcdb4708,Completed,"0,516","0,045",3.130,84,poisson,"0,659","9,751",620,


## 5. Horizon rest of month

### 5.1 The grid as a tuning job

Six combinations, every one run (Grid), 2 in parallel, 100 trees each. `max-features` 0,7 instead of 08's 0,5 (0,5
lost at every leaf, by 2 - 3 MAE and twice the bias); leaf 25 is new and the heaviest point: 2,5 M rows down to leaves
of 25 on 16 GB, which may end in a `MemoryError`. A failed point shows as a failed job in the table and the reading
records it; the grid's other points stand. The pivot shows the MAE over the two axes next to 08's grid (108,49 at
leaf 50 with every input, 100 trees).

In [6]:
base_rest = jobs.training_job_request(JOB["rest_of_month"], ROLE_ARN, REGION, BUCKET, "rest_of_month", FAMILY["rest_of_month"],
                                      STATIC_REST, SOURCE_URI, instance_type=INSTANCE, max_runtime_s=7200)
request_rest = jobs.tuning_job_request(JOB["rest_of_month"], base_rest, **SPACE_REST, strategy="Grid", max_jobs=None, max_parallel=MAX_PARALLEL)
jobs.submit(sm, request_rest)
tuning_rest = jobs.wait(sm, JOB["rest_of_month"], tuning=True)
trials_rest = jobs.trials(sm, JOB["rest_of_month"])
trials_rest.to_csv(SEARCH_DIR / "rest_of_month_trials.csv")
done_rest = trials_rest[trials_rest.status == "Completed"]
assert len(done_rest), "no grid point completed: read the failure column and the log of one job (jobs.log_tail)"
print(f"{len(trials_rest)} jobs, {len(done_rest)} completed, {fmt(trials_rest['training (s)'].sum() / 3600, '.1f')} instance-hours; "
      f"best {done_rest.MAE.iloc[0]:.3f} at 100 trees against 08's grid best 108.49")
display(trials_rest.round(3))
print("Validation MAE over the grid, 100 trees (a missing cell is a failed job)")
done_rest.pivot(index="min-samples-leaf", columns="max-features", values="MAE").round(2)

submitted tuning job citibike-r5-rest-tune
10:42:06 citibike-r5-rest-tune: InProgress
10:43:07 citibike-r5-rest-tune: InProgress inprogress 2
10:52:15 citibike-r5-rest-tune: InProgress completed 1, inprogress 2
10:53:16 citibike-r5-rest-tune: InProgress completed 2, inprogress 2
10:58:22 citibike-r5-rest-tune: InProgress completed 3, inprogress 1
10:59:23 citibike-r5-rest-tune: InProgress completed 3, inprogress 2
11:02:26 citibike-r5-rest-tune: InProgress completed 4, inprogress 2
11:07:30 citibike-r5-rest-tune: InProgress completed 5, inprogress 1
11:10:33 citibike-r5-rest-tune: Completed completed 6
6 jobs, 6 completed, 0,7 instance-hours; best 107.606 at 100 trees against 08's grid best 108.49


,status,max-features,min-samples-leaf,MAE,training (s),failure
job,,,,,,
citibike-r5-rest-tune-004-52c51655,Completed,"1,0",25,"107,606",500,
citibike-r5-rest-tune-001-073588d0,Completed,"0,7",25,"108,163",455,
citibike-r5-rest-tune-005-1937aa3e,Completed,"1,0",50,"108,489",465,
citibike-r5-rest-tune-002-4867ab59,Completed,"0,7",50,"109,267",445,
citibike-r5-rest-tune-006-1c3d7374,Completed,"1,0",100,"109,929",440,
citibike-r5-rest-tune-003-970143af,Completed,"0,7",100,"110,864",389,


Validation MAE over the grid, 100 trees (a missing cell is a failed job)


max-features,0.700000,1.000000
min-samples-leaf,,
25,"108,16","107,61"
50,"109,27","108,49"
100,"110,86","109,93"


### 5.2 The refit with 200 trees

The best point of the grid, fitted again with 200 trees as one training job (08 did the same on the laptop; more
trees only lower the variance of the average, by 0,02 there). This job's artefacts are the rest-of-month candidate of
10.

In [7]:
best_rest = done_rest.index[0]
params_rest = {"min-samples-leaf": int(done_rest.loc[best_rest, "min-samples-leaf"]),
               "max-features": float(done_rest.loc[best_rest, "max-features"]), "n-estimators": TREES_FINAL_REST}
request_refit = jobs.training_job_request(JOB["refit"], ROLE_ARN, REGION, BUCKET, "rest_of_month", FAMILY["rest_of_month"],
                                          params_rest, SOURCE_URI, instance_type=INSTANCE, max_runtime_s=7200)
jobs.submit(sm, request_refit)
refit_rest = jobs.wait(sm, JOB["refit"])
assert refit_rest["TrainingJobStatus"] == "Completed", refit_rest.get("FailureReason")
refit_metrics = {m["MetricName"]: m["Value"] for m in refit_rest.get("FinalMetricDataList", [])}
print(f"refit {params_rest}: validation MAE {refit_metrics.get('validation:MAE')} (grid point {done_rest.MAE.iloc[0]:.3f} at 100 trees; "
      f"08's refit 108.47), billed {refit_rest.get('BillableTimeInSeconds')} s")

submitted training job citibike-r5-rest-refit
11:10:34 citibike-r5-rest-refit: InProgress Starting
11:13:37 citibike-r5-rest-refit: InProgress Downloading
11:14:37 citibike-r5-rest-refit: InProgress Training
11:29:50 citibike-r5-rest-refit: InProgress Uploading
11:30:51 citibike-r5-rest-refit: Completed Completed
refit {'min-samples-leaf': 25, 'max-features': 1.0, 'n-estimators': 200}: validation MAE 107.46640014648438 (grid point 107.606 at 100 trees; 08's refit 108.47), billed 992 s


## 6. The winners back on the laptop, checked

Per horizon, the job whose artefacts become `10_<horizon>_<family>/`: the best job of the day tuning job (1.000 trees,
as trained; when several jobs are within 0,1 % of the best MAE, the one with the fewest leaves, the smaller model, as
09's rule prefers) and the refit for the rest of the month. `fetch_artefacts()` unpacks `model.tar.gz` and `output.tar.gz`
and writes `job.json` (the job, its hyperparameters, instance, image, metrics, times). Then the check 09 repeats: the
pipeline, loaded in this kernel, must reproduce the job's own `predictions_valid.parquet` on 2.000 validation rows.
It may not: the container fitted with scikit-learn 1.3 and its lightgbm, this kernel runs newer versions, and a
pickle does not always cross versions. When it does not load or differs, the configuration is **refitted here**
through the same script with the same seed (the job's metrics.json is kept as `metrics_sagemaker.json`, `job.json`
says `refit_locally` and both validation MAEs); what 10 found is the configuration, and the job's own artefacts stay
in S3.

In [ ]:
close_day = done_day[done_day.MAE <= done_day.MAE.iloc[0] * 1.001]        # within 0,1 % of the best: a tie for this search
day_winner = (close_day["num-leaves"] * STATIC_DAY["n-estimators"]).idxmin() if len(close_day) > 1 else done_day.index[0]
if len(close_day) > 1:
    print(f"day: {len(close_day)} jobs within 0,1 % of the best MAE; the one with the fewest leaves wins: {day_winner} "
          f"({int(done_day.loc[day_winner, 'num-leaves'])} leaves, MAE {done_day.loc[day_winner, 'MAE']:.4f} against {done_day.MAE.iloc[0]:.4f})")
winners = {"day": day_winner, "rest_of_month": JOB["refit"]}
df = load_features()


def check_pipeline(folder: Path, horizon: str, n: int = 2000) -> float | None:
    """The largest difference between the pipeline in folder and its saved validation predictions on n rows,
    or None when the pipeline does not load in this kernel."""
    rows = df[(df.split == "valid") & scorable(df, horizon)].sample(n, random_state=0)
    valid_inputs = pd.read_parquet(MODEL_INPUTS_DIR / horizon / "valid.parquet")
    keys = pd.MultiIndex.from_arrays([rows.station_key.astype(str), rows.day.astype("datetime64[ns]")])
    picked = valid_inputs[pd.MultiIndex.from_arrays([valid_inputs.station_key.astype(str), valid_inputs.day.astype("datetime64[ns]")]).isin(keys)]
    try:
        return float((model_predictions(folder, rows, picked) - run_predictions(folder, rows)).abs().max())
    except Exception as e:   # a pickle from other library versions fails in many ways; every one means: refit here
        print(f"{folder.name}: the pipeline does not load here ({type(e).__name__}: {str(e)[:100]})")
        return None


def refit_here(folder: Path, horizon: str, record: dict) -> dict:
    """train.py on this laptop with the job's hyperparameters and seed, into the same folder; the job's metrics kept."""
    (folder / "metrics.json").replace(folder / "metrics_sagemaker.json")
    params = {k: v for k, v in record["hyperparameters"].items() if k not in ("family", "horizon")}
    argv = ["--family", FAMILY[horizon], "--horizon", horizon, "--model-dir", str(folder),
            "--train", str(MODEL_INPUTS_DIR / horizon / "train.parquet"), "--valid", str(MODEL_INPUTS_DIR / horizon / "valid.parquet")]
    argv += [token for key, value in params.items() for token in (f"--{key}", str(value))]
    with open(folder / "refit_log.txt", "w", encoding="utf-8") as log, redirect_stdout(log):
        result = train.main(argv)
    record.update(refit_locally=True, refit_validation_mae=result["validation"]["MAE"], refit_fit_seconds=result["fit_seconds"])
    (folder / "job.json").write_text(json.dumps(record, indent=1, default=str), encoding="utf-8")
    return result


for horizon, job_name in winners.items():
    folder = RUNS_10[horizon]
    record = jobs.fetch_artefacts(s3, jobs.describe(sm, job_name), folder)
    job_mae = record["metrics"].get("validation:MAE")
    gap = check_pipeline(folder, horizon)
    if gap is not None and gap < 1e-6:
        print(f"{horizon}: {job_name} -> {folder.name}, {record['hyperparameters']}; the pipeline reproduces the job's predictions "
              f"(largest difference {gap:.1e}); validation MAE {job_mae}")
    else:
        print(f"{horizon}: {job_name} -> {folder.name}; refitting {record['hyperparameters']} here"
              + (f" (the loaded pipeline differs by up to {gap:.4f})" if gap is not None else ""))
        result = refit_here(folder, horizon, record)
        print(f"{horizon}: refit validation MAE {result['validation']['MAE']:.3f} against the job's {job_mae}, fit {result['fit_seconds']} s")

day: 2 jobs within 0,1 % of the best MAE; the one with the fewest leaves wins: citibike-r5-day-tune-012-a929d019 (501 leaves, MAE 9.5342 against 9.5336)
3.872.821 rows from data\citibike\station_day_features.parquet


c:\Users\peter.henskens\.conda\envs\DataSpell\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sagemaker-us-east-1-387312886158.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\peter.henskens\.conda\envs\DataSpell\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sagemaker-us-east-1-387312886158.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\peter.henskens\.conda\envs\DataSpell\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sagemaker-us-east-1-387312886158.s3.amazonaws.com'. Adding certificate verification is strongl

10_day_lightgbm: the pipeline does not load here (AttributeError: 'str' object has no attribute 'transform')
day: citibike-r5-day-tune-012-a929d019 -> 10_day_lightgbm; refitting {'colsample-bytree': 0.8422629782115849, 'curve-every': 100, 'family': 'lightgbm', 'horizon': 'day', 'learning-rate': 0.02168462718600527, 'min-child-samples': 100, 'n-estimators': 1000, 'num-leaves': 501, 'objective': 'regression_l1', 'subsample': 0.6364242375286887} here
day: refit validation MAE 9.534 against the job's 9.534199714660645, fit 186.6 s


c:\Users\peter.henskens\.conda\envs\DataSpell\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sagemaker-us-east-1-387312886158.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\peter.henskens\.conda\envs\DataSpell\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sagemaker-us-east-1-387312886158.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\peter.henskens\.conda\envs\DataSpell\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sagemaker-us-east-1-387312886158.s3.amazonaws.com'. Adding certificate verification is strongl

10_rest_of_month_extra_trees: the pipeline does not load here (AttributeError: 'str' object has no attribute 'transform')
rest_of_month: citibike-r5-rest-refit -> 10_rest_of_month_extra_trees; refitting {'family': 'extra_trees', 'horizon': 'rest_of_month', 'max-features': 1.0, 'min-samples-leaf': 25, 'n-estimators': 200} here
rest_of_month: refit validation MAE 107.466 against the job's 107.46640014648438, fit 379.8 s


: 

## 7. Reading and what carries forward

Run of 10 October 2026 (team lead), from the laptop against the AWS Academy Learner Lab, in one session: the smoke job
at 08:51, the day tuning job 08:58 - 10:25, the grid 10:26 - 11:10, the refit 11:11 - 11:31, the artefacts back on the
laptop just after. Every job completed; nothing was tuned or changed after the trial tables were read, except the
tie-break of part 6. Validation 2025, MAE in departures per station-day, the same rows and script as 07 and 08:

| Horizon | 08 (validation MAE, laptop) | 10, best trial | 10, the candidate | Where the tuner went | Jobs, instance-hours, cost |
| --- | --- | --- | --- | --- | --- |
| day | 9,634 (500 trees, 255 leaves, 12 MB) | 9,5336 (1.017 leaves) and 9,5342 (501 leaves): a tie | **9,534** (WAPE 16,3 %, bias -1,0): absolute error, 501 leaves, learning rate 0,022, leaf floor 100, 1.000 trees, **46 MB** | past 08's edges: 501 - 1.023 leaves in the top three, 1.000 trees, the absolute error in the five best | 12 jobs of 5 - 23 min, 2,5 instance-hours, about 0,6 $ |
| rest of month | 108,47 (leaf 50, 1,0 GB) | 107,61 at 100 trees (leaf 25, every input) | **107,47** (WAPE 11,7 %, bias -4,6): leaf 25, every input, 200 trees, **2,0 GB** | one step below 08's smallest leaf; every input beats 0,7 of them at every leaf | 6 + 1 jobs of 6 - 17 min, 1,0 instance-hour, about 0,25 $ |

- **The widened spaces paid, by about 1 % each.** Day: 9,634 -> 9,534, with the tuner choosing exactly what 08's
  edges suggested: more leaves (501 and 1.017 instead of 255), more trees (1.000, the curve's best point is still
  the last one) and the absolute error, which the five best trials share while the best squared error sits at
  9,696 and the best Poisson at 9,732. The leaf floor hardly matters (142 and 792 at 1.017 leaves: 9,534 and 9,566).
  Rest of month: 108,47 -> 107,47, one halving of the leaf further; the gain per halving stays about 0,9, and the
  model doubles each time.
- **The same script gives the same model on both machines.** The two grid points 08 also ran, (50, 1,0) and
  (100, 1,0), came back from the container as 108,489 and 109,929: 08's laptop numbers to the third decimal, with
  scikit-learn 1.3.2 there against 1.9.1 here. And the refits of part 6 reproduce the jobs' validation MAE to the
  sixth decimal (9,534226 and 107,46643) with the same seed. 07's design decision, one `train.py` for the laptop
  and SageMaker, is measured rather than assumed.
- **The pickles did not cross the versions, the configurations did.** Neither pipeline from the container
  (scikit-learn 1.3.2) loads under 1.9.1 (the ColumnTransformer's pickle changed), so part 6 refitted both here:
  187 s for the LightGBM (244 s in the container), 380 s for the extra trees (802 s there, 4 vCPU against the
  laptop's). `job.json` records both numbers; the jobs' own artefacts stay in S3. What 10 contributes is the
  configuration, found on AWS, and that is also what a retraining pipeline needs. The kernel died right after the
  second refit had written its files (the 2 GB model next to the 1,2 GB feature table on a 16 GB laptop); nothing
  was lost, every file of part 6 is on disk.
- **A tie, broken by size.** Trials 002 and 012 differ by 0,0006 in MAE and by a factor two in leaves; the
  candidate is the smaller one (501 leaves, 46 MB), on the same principle as 09's rule. With the strict "lowest
  MAE" the 1.017-leaf model would have gone to 09 with the same score.
- **The container cost four smoke jobs before the fifth passed.** Its packages are installed with conda, and pip's
  replacement of any compiled one (scipy, then pyarrow) leaves files behind that the new version loads; the fix is
  to pin numpy, scipy, pandas and pyarrow to the container's own versions and to add only scikit-learn 1.3.2 and
  lightgbm. The lesson for 10's appendix and for anyone reusing the script: read the whole job log, not its tail.
- **Cost and time.** About 4 instance-hours of ml.m5.xlarge in all, the failed smoke jobs included: around 1 $
  at the list price, against a lab budget of tens of dollars. Wall time of the successful run 2 h 40 min, with two
  jobs in parallel; the session timer was refreshed once.

**What 09 takes from here:** the two run folders `10_day_lightgbm` and `10_rest_of_month_extra_trees`, scored on
the test period by running 09 again, with the rule 09 fixed before any test number was seen. **What the
deployment takes:** the trial tables as the record of the search, `job.json` as the record of where each candidate
was found, `jobs.py` as the way a retraining pipeline submits the same script to SageMaker; and a size question
that 10 made sharper, not smaller: the day model is 46 MB (08: 12 MB) and the rest-of-month model 2,0 GB (08:
1,0 GB, 07: 0,5 GB) for about 1 % less error each.

## Appendix: the dry run of 8 October, the expected runtime and cost, the failures to expect

**Dry run** (Claude, 8 October, this kernel's environment, before any job was submitted). Two checks, neither touching AWS. (1) `citibike/tests/test_jobs.py`, 11 tests: the training and tuning requests validate against the AWS API model of boto3 (every field, every type), the hyperparameters survive the container's JSON decoding, the Grid request carries no job count, the metric regex reads the script's line, the bundle holds the two files, the artefacts unpack and the trial table sorts and decodes. (2) Every cell executed by `nbclient` in the kernel's environment with `boto3.Session` replaced by a fake AWS (STS, S3, SageMaker and CloudWatch Logs answered from the fixtures of 09's dry run: the 75-station sample, its model inputs, its run folders) on the dry-run sample: 15 s, no error, every assertion passed. The fake returned 12 day trials (one failed) and 6 grid points (leaf 25 with every input failed with a MemoryError, as the appendix expects), the day winner's model archive was made unloadable on purpose, so part 6 took the refit branch for the day (the configuration refitted here through `train.py`, `job.json` records it) and the check branch for the rest of the month (the pipeline reproduced the job's predictions to 10^-11). **Not proven by the dry run**: that the container's Python is 3.8 and installs the requirements (the smoke job proves it: 10 October, Python 3.9 and the numpy pin), that the lab allows tuning jobs (the first submit proves it), the runtimes and the cost.

**Expected runtime and cost**: an extrapolation, not a measurement (the measured times and billed seconds are in the
outputs of parts 3 - 5 after the run). The upload of 230 MB a few minutes; the smoke job 3 - 5 minutes (two of them
the container start and the package install); the day tuning job 12 jobs of 5 - 10 minutes, two at a time, about
an hour; the grid 6 jobs of 10 - 25 minutes, two at a time, about an hour; the refit 15 - 30 minutes. About
**5 instance-hours of ml.m5.xlarge**, a few dollars at the list price, in two to three hours of wall time: one lab
session if it is started right before part 3, two otherwise. **Measured in the run of 10 October:** the upload 228 MB in 77 s; the smoke job 104 billed seconds (the script 3 s of it); the 12 day jobs 320 - 1.387 s each, 2,5 instance-hours, 87 min of wall time with two in parallel; the 6 grid jobs 389 - 500 s, 0,7 instance-hours, 44 min; the refit 992 s; part 6 with both local refits about 12 min. About 4 instance-hours in all, the four failed smoke jobs included, within one lab session of 2 h 40 min for the successful run.

| Symptom | Cause | Fix |
| --- | --- | --- |
| `SSLError ... self-signed certificate in certificate chain` on the first S3 call | the laptop's TLS-inspecting proxy; Python's CA bundle does not hold its certificate | the first two lines of the setup cell (`truststore`); restart the kernel if `boto3` was imported before them |
| `RecursionError` inside botocore after a request | `truststore.inject_into_ssl()` ran after `boto3` had been imported in this kernel | restart the kernel and run the setup cell first |
| the assertion in part 0, or `NoCredentialsError` | no credentials in `.env`, or the lab is not started | start the lab (the dot turns green), AWS Details → AWS CLI → Show, the three values into the repo's `.env` as `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN` |
| `the lab session's credentials expired` in a wait | the four hours are over | start the lab again, put the new values in `.env`, run the setup cell (it reloads the file), run the waiting cell again: the job is found by name |
| `InvalidClientTokenId` or `UnrecognizedClientException` on the first call | the three values in `.env` are from different sessions, or one was pasted incompletely (the session token is one long line) | copy the whole block again |
| `AccessDeniedException ... not authorized to perform: iam:PassRole on ... role/LabRole` | this lab's policy does not let you hand LabRole to SageMaker (the challenge lab did, 8 October) | use the role the lab's own SageMaker resources run with: the setup cell lists them ("roles this account already hands to SageMaker"), or the console (the *AWS* button) → SageMaker → Notebook instances → the instance's IAM role ARN; put that ARN in `.env` as `SAGEMAKER_ROLE_ARN` and run the setup cell again. If no role can be passed from the laptop, run this notebook inside that notebook instance (upload `citibike/train/` and this notebook; the instance's role passes itself) |
| `AccessDeniedException` on a create call naming the instance type | not in the lab's list (ml.t3.medium/large/xlarge, ml.m5.large/xlarge, ml.c5.large/xlarge) | keep `INSTANCE = "ml.m5.xlarge"` |
| the session timer nears 0:00 while a tuning job runs | sessions are limited | choose *Start Lab* again before it ends: the timer is refreshed and the credentials stay valid; the jobs run on regardless |
| `ResourceLimitExceeded` | the account's quota for ml.m5.xlarge training jobs is below `MAX_PARALLEL` | `MAX_PARALLEL = 1`; the lab cannot raise quotas |
| the smoke job fails with `... failed to import` (numpy, `_multiarray_umath`) or a `TypeError` inside scipy at import | pip replaced a conda-installed compiled package of the container (numpy, scipy, pandas or pyarrow) and its compiled files stayed behind (four jobs on 10 October) | keep numpy, scipy, pandas and pyarrow pinned to the container's own versions in `citibike/train/requirements.txt`; after a change: part 2 again (the bundle is re-uploaded) and a new `RUN_TAG`. The cell prints 80 log lines on a failure: the pip lines say what was replaced ("Attempting uninstall"), the traceback which import failed |
| the smoke job fails in the package install (the log tail shows pip) | `citibike/train/requirements.txt` asks a version the container's Python 3.8 cannot install | relax the pin, run part 2 again (the bundle is re-uploaded) and part 3 with a new `RUN_TAG` |
| the smoke job fails in the script (the log tail shows a traceback) | a difference between the laptop and the container | fix `train.py`, the dry run of 07 is the local check; part 2 and 3 again with a new `RUN_TAG` |
| a grid point fails with `MemoryError` | leaf 25 with every input on 16 GB | expected risk; the other points stand; the reading records it |
| "exists already: read back" for a job you meant to run anew | job names are unique per account and region | bump `RUN_TAG` |
| `AccessDeniedException` on `CreateHyperParameterTuningJob` | the lab policy does not allow tuning jobs | the fallback: submit the configurations as training jobs with `jobs.training_job_request()` in a loop (08's draw as the list) and read them back with `jobs.training_jobs_table()`; the rest of the notebook reads the same table |
| "the pipeline does not load here" in part 6 | a pickle across scikit-learn or lightgbm versions | expected; part 6 refits the configuration here and records it in `job.json` |
| the kernel dies at the end of part 6 (the IDE reports a crash after the last printed line) | the 2 GB extra trees refit next to the feature table on 16 GB (seen 10 October) | everything is on disk already (`job.json`, `metrics.json`, `model.joblib`); restart the kernel; a rerun of part 6 repeats the downloads and the refits |
| the Studio or console shows nothing under SageMaker | the console is on another region | switch the console to us-east-1 (N. Virginia) |